In [ ]:
import os, time, numpy as np, gymnasium as gym, torch
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy

os.makedirs("logs_dqn", exist_ok=True)
os.makedirs("logs_ppo", exist_ok=True)
os.makedirs("logs_a2c", exist_ok=True)

print("CUDA:", torch.cuda.is_available())

Para cada una de las opciones vamos a probar con unas mini variaciones (aunque se tengan los hiperparametros tuneadis)

In [ ]:
import statistics

def mini_crosval(algo_name, steps, log_dir, LRs, algo_class, hiperparametros):
    LRs = LRs
    SEEDS = [0, 42]
    cv_steps = steps
    cv_dir = f"logs_{algo_name}{log_dir}"

    os.makedirs(cv_dir, exist_ok=True)
    sweep_results = {}

    for lr in LRs:
        for seed in SEEDS:
            run_name = f"lr{lr:.0e}_s{seed}"
            run_dir  = f"{cv_dir}/{run_name}"
            os.makedirs(run_dir, exist_ok=True)

            env = Monitor(gym.make("LunarLander-v3"), filename=f"{run_dir}/train")
            eval_env = Monitor(gym.make("LunarLander-v3"), filename=f"{run_dir}/eval")
            
            # MISMA semilla eval todas las runs
            eval_env.reset(seed=1234)   

            model = algo_class(
                "MlpPolicy", env, verbose=0, seed=seed, device="cpu",
                learning_rate=lr,
                **hiperparametros
            )

            t0 = time.time()
            model.learn(total_timesteps=cv_steps)
            dt = (time.time() - t0) / 60

            mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
            sweep_results[(lr, seed)] = {"mean": mean_r, "std": std_r, "time_min": dt}
            print(f"{run_name:18s} reward = {mean_r:7.1f} +- {std_r:5.1f}   ({dt:4.1f} min)")
            env.close(); eval_env.close()
    
    print(f"\nResumen {algo_name} por LR (media semillas)")
    agg = {}
    for lr in LRs:
        means = [sweep_results[(lr, s)]["mean"] for s in SEEDS]
        m, s = statistics.mean(means), (statistics.stdev(means) if len(means) > 1 else 0.0)
        agg[lr] = (m, s)
        print(f"lr={lr:.0e} reward = {m:7.1f} +- {s:5.1f}")

    # Gana mejor media pero penaliza mucha varianza
    best_lr = max(agg, key=lambda l: agg[l][0] - agg[l][1])
    print(f"\nGanador: lr = {best_lr:.0e} (mean-std = {agg[best_lr][0]-agg[best_lr][1]:.1f})")

    return best_lr

In [ ]:
env = Monitor(gym.make("LunarLander-v3"), filename="logs_dqn/train")
eval_env = Monitor(gym.make("LunarLander-v3"), filename="logs_dqn/eval")

model = DQN(
    "MlpPolicy", env, verbose=0, seed=42, device="cpu",
    learning_rate=6.3e-4,
    batch_size=128,
    buffer_size=50_000,
    learning_starts=0,
    gamma=0.99,
    target_update_interval=250,
    train_freq=4,
    gradient_steps=-1,
    exploration_fraction=0.12,
    exploration_final_eps=0.1,
    policy_kwargs=dict(net_arch=[256, 256]),
)

eval_cb = EvalCallback(eval_env, best_model_save_path="logs_dqn/best",
                       log_path="logs_dqn/eval", eval_freq=5000,
                       n_eval_episodes=5, deterministic=True, verbose=0)

t0 = time.time()
model.learn(total_timesteps=100_000, callback=eval_cb)
print(f"DQN entrenamiento: {(time.time()-t0)/60:.1f} min")

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"DQN reward 20 episodios: {mean_r:.1f} +- {std_r:.1f}")
model.save("logs_dqn/dqn_lunarlander")

env.close()
eval_env.close()

In [ ]:
env = Monitor(gym.make("LunarLander-v3"), filename="logs_ppo/train")
eval_env = Monitor(gym.make("LunarLander-v3"), filename="logs_ppo/eval")

model = PPO(
    "MlpPolicy", env, verbose=0, seed=42, device="cpu",
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    learning_rate=3e-4,
)

eval_cb = EvalCallback(eval_env, best_model_save_path="logs_ppo/best",
                       log_path="logs_ppo/eval", eval_freq=5000,
                       n_eval_episodes=5, deterministic=True, verbose=0)

t0 = time.time()
model.learn(total_timesteps=500_000, callback=eval_cb)
print(f"PPO entrenamiento: {(time.time()-t0)/60:.1f} min")

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"PPO reward 20 episodios: {mean_r:.1f} +- {std_r:.1f}")
model.save("logs_ppo/ppo_lunarlander")

env.close()
eval_env.close()

In [ ]:
env = Monitor(gym.make("LunarLander-v3"), filename="logs_a2c/train")
eval_env = Monitor(gym.make("LunarLander-v3"), filename="logs_a2c/eval")

model = A2C(
    "MlpPolicy", env, verbose=0, seed=42, device="cpu",
    n_steps=5,
    gamma=0.995,
    gae_lambda=1.0,
    ent_coef=0.01,
    learning_rate=7e-4,
    use_rms_prop=True,
)

eval_cb = EvalCallback(eval_env, best_model_save_path="logs_a2c/best",
                       log_path="logs_a2c/eval", eval_freq=5000,
                       n_eval_episodes=5, deterministic=True, verbose=0)

t0 = time.time()
model.learn(total_timesteps=500_000, callback=eval_cb)
print(f"A2C entrenamiento: {(time.time()-t0)/60:.1f} min")

mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(f"A2C reward 20 episodios: {mean_r:.1f} +- {std_r:.1f}")
model.save("logs_a2c/a2c_lunarlander")
env.close(); eval_env.close()